# 6.2 교차검증: \(\lambda\)를 데이터로 정한다 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter06_2_cross_validation.ipynb)

책 본문: [6.2 교차검증: \(\lambda\)를 데이터로 정한다](https://smhanlab.com/book-ml/kor/ml1/chapter06/2.html)

이 노트북은 책 6.2절의 핵심 주장들을 **모두 코드로 검증**합니다:

1. `k_fold_split`의 인덱스 분할이 "각 샘플이 정확히 한 번씩 검증"이 되는지(본문 표 검증)
2. 본문 "손으로 한 번"의 5-fold 평균모델 계산이 코드와 일치하는지(12.75 vs 8.25)
3. 검증 MSE의 \(\lambda\) 곡선이 U자형인지, **fold 간 분산**이 어떤 \(\lambda\)에서 큰지
4. 섞지 않은(정렬된) 데이터로 5-fold를 돌리면 fold별 성능이 어떻게 깨지는지
5. 최적화 단계 자체가 후보 선택에 쓰인 검증값(낙관적 추정)이 진짜 성능을 어떻게 왜곡하는지 — nested CV와 비교

## 1. 본문의 순수 파이썬 함수 재현 + 인덱스 분할 검증

본문 표(12개 샘플, $k=4$)가 실제로 "각 인덱스가 정확히 한 번씩 val"이 되는지
코드 그대로 검증합니다.

In [1]:
def ridge_gradient_descent(X, y, lam, alpha, epochs):
    m, n = len(X), len(X[0])
    w = [0.0] * (n + 1)
    for _ in range(epochs):
        grad = [0.0] * (n + 1)
        for i in range(m):
            pred = w[0] + sum(w[j+1] * X[i][j] for j in range(n))
            error = pred - y[i]
            grad[0] += error
            for j in range(n):
                grad[j+1] += error * X[i][j]
        for j in range(n + 1):
            reg = lam * w[j] if j > 0 else 0
            w[j] -= alpha * (grad[j] / m + reg)
    return w

def k_fold_split(data, k, fold_idx):
    n = len(data)
    fold_size = n // k
    start = fold_idx * fold_size
    end = start + fold_size if fold_idx < k - 1 else n
    val = data[start:end]
    train = data[:start] + data[end:]
    return train, val

# 본문 표 검증: 12개 샘플(k=4)에서 각 fold의 val 인덱스
data = list(range(12))
for fold in range(4):
    tr, va = k_fold_split(data, 4, fold)
    val_idx = [d for d in va]
    print(f"fold {fold}: val={val_idx}  train={tr}")

# 각 인덱스가 정확히 한 번씩 val에 등장하는지
from collections import Counter
cnt = Counter(d for fold in range(4) for d in k_fold_split(data, 4, fold)[1])
assert all(cnt[i] == 1 for i in range(12)), cnt
print("\n인덱스 0~11 각각이 정확히 한 번씩 검증용으로 쓰인다 -> 본문의 설명과 일치")

fold 0: val=[0, 1, 2]  train=[3, 4, 5, 6, 7, 8, 9, 10, 11]
fold 1: val=[3, 4, 5]  train=[0, 1, 2, 6, 7, 8, 9, 10, 11]
fold 2: val=[6, 7, 8]  train=[0, 1, 2, 3, 4, 5, 9, 10, 11]
fold 3: val=[9, 10, 11]  train=[0, 1, 2, 3, 4, 5, 6, 7, 8]

인덱스 0~11 각각이 정확히 한 번씩 검증용으로 쓰인다 -> 본문의 설명과 일치


## 2. "손으로 한 번" 검증: 평균모델(y=1~10)의 5-fold CV

가장 단순한 모델(훈련집단 평균을 되뇔 뿐)로, \(y=1,\dots,10\)을 순서대로
2개씩 5등분한 5-fold CV를 계산합니다. 손 계산 테이블(본문과 동일)을 재현하고,
**CV 추정값(12.75)이 진짜 평균 5.5 기준 오차(8.25)보다 얼마나 큰지** 봅니다 —
검증셋이 학습에 영향을 받으면(이 경우는 훈련집단 평균이 검증 위치에 따라
5.0~6.5로 흔들려) CV 추정 자체가 실제보다 큰 편향이 생길 수 있음을 보여줍니다.

In [2]:
ys10 = list(range(1, 11))
folds = []
for fold in range(5):
    start, end = fold * 2, fold * 2 + 2
    val = ys10[start:end]
    train = ys10[:start] + ys10[end:]
    mu = sum(train) / len(train)
    err = sum((v - mu) ** 2 for v in val) / 2
    folds.append(err)
    print(f"fold {fold}: val={val}  train_mean={mu:.2f}  fold_MSE={err:.4f}")
cv = sum(folds) / 5
ideal = sum((v - 5.5) ** 2 for v in ys10) / 10
print(f"\nCV 추정값 = {cv:.4f}   (손 계산: 25.25, 6.50, 0.25, 6.50, 25.25 -> 평균 12.75)")
print(f"정직한 평균모델 오차 (진짜 평균 5.5 기준) = {ideal:.4f}")
assert cv == 12.75 and ideal == 8.25

fold 0: val=[1, 2]  train_mean=6.50  fold_MSE=25.2500
fold 1: val=[3, 4]  train_mean=6.00  fold_MSE=6.5000
fold 2: val=[5, 6]  train_mean=5.50  fold_MSE=0.2500
fold 3: val=[7, 8]  train_mean=5.00  fold_MSE=6.5000
fold 4: val=[9, 10]  train_mean=4.50  fold_MSE=25.2500

CV 추정값 = 12.7500   (손 계산: 25.25, 6.50, 0.25, 6.50, 25.25 -> 평균 12.75)
정직한 평균모델 오차 (진짜 평균 5.5 기준) = 8.2500


### (본문 §2와 같은 예) 무작위 5-fold로 다시: 평균모델 CV는 왜 8.25보다 큰가

위 §2는 *순서대로* 2개씩 잘랐을 때의 예다. 본문 §손으로한번에서는 실제로 무작위 5-fold를
돌렸을 때의 fold별 값을 (시드 고정으로) 함께 보여준다 — fold마다 검증쌍이 달라지므로
fold별 MSE가 1.39~20.14까지 흩어지고, 평균은 10.39로 손으로 계산한 12.75와는
다르지만 정직한 바닥 8.25보다 크다. "모델이 아무것도 못 배운" 예제에서도 CV가
진짜 오차보다 높게 나온다는 점이 공통 메시지다.

In [3]:
import numpy as np
import random
rng = np.random.default_rng(7)
perm = rng.permutation(10)
ys10 = np.arange(1, 11, dtype=float)
errs = []
for fold in range(5):
    val_idx = [int(i) for i in sorted(np.where(perm % 5 == fold)[0])]   # 이 fold의 val에 들어가는 원본 인덱스
    val = ys10[val_idx]
    mu = (ys10.sum() - val.sum()) / 8.0
    e = float(((val - mu) ** 2).mean())
    errs.append(e)
    print(f"fold {fold}: val={val_idx}  train_mean={mu:.2f}  fold_MSE={e:.4f}")
cv = float(np.mean(errs))
ideal = float(((ys10 - ys10.mean()) ** 2).mean())
print(f"\nCV 추정값 = {cv:.4f}   (정직한 평균모델 오차 8.25의 {cv/ideal:.2f}배)")
assert abs(cv - 10.3875) < 1e-4

fold 0: val=[1, 8]  train_mean=5.50  fold_MSE=12.2500
fold 1: val=[3, 5]  train_mean=5.62  fold_MSE=1.3906
fold 2: val=[2, 6]  train_mean=5.62  fold_MSE=4.3906
fold 3: val=[0, 4]  train_mean=6.12  fold_MSE=13.7656
fold 4: val=[7, 9]  train_mean=4.62  fold_MSE=20.1406

CV 추정값 = 10.3875   (정직한 평균모델 오차 8.25의 1.26배)


## 3. U자형 곡선: 검증 MSE를 \(\lambda\)에 대해 그린다

15개 특징 중 2개만 진짜 관련 있는 데이터($n=100$, 잡음 표준편차 3)로, 5-fold CV를
로그 간격 41개 \(\lambda\) 후보에 대해 돌립니다. Ridge는 정규방정식으로
닫혀 있으므로(6.1절 연습문제 3) 경사하강 수렴 걱정이 없습니다 — **완전 수렴**한
해로만 봅니다.

각 \(\lambda\)마다 **5개 fold별 MSE 점**과 **평균 ± 표준편차 밴드**를 함께
그림으로써 "검증 MSE 평균이 U자형"이라는 주장 뒤에 숨은 fold 간 변동도 함께
확인합니다.

In [4]:
import numpy as np

def fit_ridge(X, y, lam):   # 6.1절 연습문제 3: bias는 정규화에서 제외(centering trick)
    x0, y0 = X.mean(0), y.mean()
    Xc, yc = X - x0, y - y0
    w = np.linalg.solve(Xc.T @ Xc + lam * np.eye(Xc.shape[1]), Xc.T @ yc)
    return w, y0 - w @ x0

def score(w, b, X, y):
    return float(((X @ w + b - y) ** 2).mean())

def cv_errors(X, y, lam, k=5):
    m = len(y)
    errs = []
    for i in range(k):
        va = np.arange(i, m, k)
        tr = np.concatenate([np.arange(j, m, k) for j in range(k) if j != i])
        w, b = fit_ridge(X[tr], y[tr], lam)
        errs.append(score(w, b, X[va], y[va]))
    return errs

rng = np.random.default_rng(11)
n, d = 100, 15
X = rng.normal(0, 1, (n, d))
y = 3 * X[:, 0] - 2 * X[:, 4] + rng.normal(0, 3.0, n)   # 2개만 관련, 잡음 sd=3

lams = np.logspace(-2, 2, 41)
all_errs = np.array([cv_errors(X, y, lam) for lam in lams])
means, stds = all_errs.mean(1), all_errs.std(1)
best = int(np.argmin(means))
print(f"최적 lambda = {lams[best]:.3f}  (5-fold 평균 검증 MSE = {means[best]:.3f})")
for j in [0, 10, best, 30, 40]:
    print(f"lambda={lams[j]:9.4f}: mean={means[j]:7.3f}  std={stds[j]:6.3f}  folds={['%.2f' % e for e in all_errs[j]]}")

최적 lambda = 15.849  (5-fold 평균 검증 MSE = 9.031)
lambda=   0.0100: mean=  9.888  std= 3.220  folds=['5.50', '7.27', '11.40', '14.68', '10.58']
lambda=   0.1000: mean=  9.875  std= 3.201  folds=['5.51', '7.28', '11.37', '14.64', '10.58']
lambda=  15.8489: mean=  9.031  std= 1.663  folds=['6.57', '8.22', '8.60', '10.72', '11.05']
lambda=  10.0000: mean=  9.125  std= 1.959  folds=['6.18', '7.86', '9.19', '11.60', '10.79']
lambda= 100.0000: mean= 11.138  std= 2.329  folds=['10.44', '12.51', '8.61', '9.18', '14.95']


In [5]:
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.semilogx(lams, means, "-o", ms=3, lw=2, label="5-fold 평균 검증 MSE")
ax.fill_between(lams, means - stds, means + stds, alpha=0.25, label="±1 표준편차 (fold 간)")
for col in all_errs.T:
    ax.semilogx(lams, col, ":", lw=0.8, color="0.55")
ax.plot([], [], ":", lw=0.8, color="0.55", label="각 fold별 MSE")
ax.axvline(lams[best], color="red", ls="--", lw=1)
ax.text(lams[best] * 1.15, ax.get_ylim()[0] + 0.3, f"$\\lambda^*\\approx{lams[best]:.1f}$",
        color="red", fontsize=11)
ax.set_xlabel("$\\lambda$ (로그 스케일)")
ax.set_ylabel("검증 MSE (평균)")
ax.set_title("$\\lambda$에 따른 검증 MSE — U자형(좌: 과적합 쪽, 우: 과소적합 쪽)")
ax.legend(fontsize=9, loc="upper left")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG + "/ch06_2_lambda_ucurve.svg")
plt.show()
print("figure saved -> kor/src/images/ch06_2_lambda_ucurve.svg")

figure saved -> kor/src/images/ch06_2_lambda_ucurve.svg


## 4. $\lambda^*$는 데이터가 바뀌면 어떻게 흔들리는가

같은 생성 과정(2개 관련 특징 + 13개 잡음 특징, 잡음 sd=3)에서 시드만 바꾸면
\(\lambda^*\)는 어디에 모이는지 10개 시드에서 확인합니다. "최적점" 자체가
추정치이므로 데이터에 따라 흔들릴 수 있고, **인접한 \(\lambda\)들도
성능 차이가 작다면** 그 범위를 통째로 '괜찮은 영역'으로 봐야 한다는
실용적 교훈을 숫자로 확인합니다.

In [6]:
print("seed | best lambda | 해당 5-fold 평균 MSE")
bests = []
for seed in range(1, 11):
    r = np.random.default_rng(seed)
    Xs = r.normal(0, 1, (100, 15))
    ys = 3 * Xs[:, 0] - 2 * Xs[:, 4] + r.normal(0, 3.0, 100)
    ms = np.array([np.mean(cv_errors(Xs, ys, lam)) for lam in lams])
    b = int(np.argmin(ms))
    bests.append(lams[b])
    print(f"{seed:4d} |  {lams[b]:8.3f}    | {ms[b]:.3f}")
print(f"\n10개 시드의 best lambda: min={min(bests):.2f}, max={max(bests):.2f}")

seed | best lambda | 해당 5-fold 평균 MSE
   1 |     6.310    | 11.509
   2 |    15.849    | 12.796
   3 |    10.000    | 8.128
   4 |    19.953    | 11.017
   5 |    12.589    | 12.190
   6 |     5.012    | 11.117
   7 |    12.589    | 10.763
   8 |    10.000    | 12.271
   9 |     5.012    | 8.279
  10 |    10.000    | 8.687

10개 시드의 best lambda: min=5.01, max=19.95


### 1SE 규칙 (Tibshirani): '최소점 ±1 표준편차' 안의 가장 단순한 모델
본문 §1SE 규칙의 내용을 같은 곡선 위에서 계산해본다 — 최소점(λ≈15.8, MSE 9.03)보다 한 표준편차(1.66) 높은 10.69까지 허용하면, 그 안의 가장 단순한 모델은λ≈79.4(MSE 10.59)까지 간다. 밴드가 최소점의 5배까지 열려 있다는 것은"제일 낮은 한 점"을 그대로 믿지 말라는 실용적 경고다.

In [7]:
thresh = means[best] + stds[best]
ok = int(max(np.where(means <= thresh)[0]))
print(f"1SE 기준: thresh = {means[best]:.3f} + {stds[best]:.3f} = {thresh:.3f}")
print(f"1SE 안의 가장 단순한 모델: lambda = {lams[ok]:.1f} (5-fold 평균 MSE = {means[ok]:.3f})")
print(f"-> lambda* (최소점) = {lams[best]:.1f} 대비 {lams[ok]/lams[best]:.1f}배까지 허용되는 구간")


1SE 기준: thresh = 9.031 + 1.663 = 10.694
1SE 안의 가장 단순한 모델: lambda = 79.4 (5-fold 평균 MSE = 10.594)
-> lambda* (최소점) = 15.8 대비 5.0배까지 허용되는 구간


## 5. "자주 하는 실수" 정량화: 섞지 않은(정렬된) 데이터의 5-fold

분류 문제에서 샘플이 클래스순으로 정렬되어 있으면(앞 15개 y=0, 뒤 15개 y=1),
`k_fold_split`이 주어진 순서 그대로 자르는 순간 fold별 클래스 비율이
0/100에서 100/0까지 벌어집니다. **평균 검증 정확도는 섞은 것과 비슷하게
보여도** fold별 값이 1.0에서 0.0까지 흔들린다면, 그 "평균"은 어떤 \(\lambda\)
도 좋게 보이게 만드는 근거 없는 숫자일 수 있습니다.

In [8]:
import random
n = 30
Xs = [[float(i) - 50.0] for i in range(15)] + [[100.0 + float(i) - 50.0] for i in range(15)]
ys = [0.0] * 15 + [1.0] * 15

def cv_acc(order, k=5, lam=0.0):
    data = [(Xs[i], ys[i]) for i in order]
    accs = []
    for fold in range(k):
        tr, va = k_fold_split(data, k, fold)
        Xtr, ytr = [d[0] for d in tr], [d[1] for d in tr]
        Xva, yva = [d[0] for d in va], [d[1] for d in va]
        w = ridge_gradient_descent(Xtr, ytr, lam, 0.01, 2000)
        correct = sum(1 for x, yv in zip(Xva, yva)
                      if (w[0] + w[1] * x[0] > 0) == (yv > 0))
        accs.append(correct / len(yva))
    return accs

sorted_accs = cv_acc(list(range(n)))
perm = list(range(n)); random.Random(42).shuffle(perm)
shuf_accs = cv_acc(perm)
print(f"정렬(안 섞음): fold별 acc = {[f'{a:.2f}' for a in sorted_accs]}  mean={np.mean(sorted_accs):.2f}")
print(f"섞음:         fold별 acc = {[f'{a:.2f}' for a in shuf_accs]}  mean={np.mean(shuf_accs):.2f}")
print(f"\n정렬: fold별 표준편차={np.std(sorted_accs):.3f}   섞음: {np.std(shuf_accs):.3f}")

정렬(안 섞음): fold별 acc = ['1.00', '1.00', '0.50', '0.00', '0.00']  mean=0.50
섞음:         fold별 acc = ['0.50', '0.67', '0.33', '0.33', '0.67']  mean=0.50

정렬: fold별 표준편차=0.447   섞음: 0.149


## 6. 최적화 편향: 단일 분할 vs LOOCV vs nested LOOCV

작은 예제($m=10$, 선형, $\lambda \in \{0, 1, 10\}$)에서 세 가지 "최종 성능"
추정치를 비교합니다:

- **단일 7/3 분할**: train 7개로 $\lambda$를 골라 test 3개에서 재는 값
- **LOOCV 곡선**: 각 $\lambda$의 LOOCV 평균 중 가장 작은 값 (낙관적: 곡선을
  만드는 최적화 단계가 같은 데이터를 또 보았다)
- **nested LOOCV**: 바깥에서 1개를 빼고, *남은 9개로만* LOOCV를 돌려
  $\lambda$를 고른 뒤 그 1개를 예측 — 최적화 단계가 평가 데이터에 닿지 않음

세 값이 "진짜" 성능(이 예제의 노이즈 플로어 ≈ 잡음 분산)보다 **어느 쪽으로
편향**되어 있는지 봅니다.

In [9]:
rng = np.random.default_rng(3)
xt = np.arange(10.0)
yt = 2.0 * xt + np.array([0.0, 1.5, 0.2, -1.0, 0.5, 1.0, -0.5, 0.3, 1.2, -1.8])
Xt, Yt = xt.reshape(-1, 1), yt
grid = [0.0, 1.0, 10.0]
noise_var = float(((yt - 2 * xt) ** 2).mean())

# (a) 단일 7/3 분할
best_err, best_lam = 1e9, None
for lam in grid:
    w, b = fit_ridge(Xt[:7], Yt[:7], lam)
    e = score(w, b, Xt[7:], Yt[7:])
    if e < best_err:
        best_err, best_lam = e, lam

# (b) LOOCV 곡선
def loocv_curve(X, y, grid):
    out = {}
    for lam in grid:
        errs = []
        for i in range(len(y)):
            tr = np.concatenate([np.arange(i), np.arange(i + 1, len(y))])
            w, b = fit_ridge(X[tr], y[tr], lam)
            errs.append(score(w, b, X[i:i+1], y[i:i+1]))
        out[lam] = float(np.mean(errs))
    return out

curve = loocv_curve(Xt, Yt, grid)
loocv_best = min(curve.values())

# (c) nested LOOCV
nested = []
picks = []
for i in range(len(Yt)):
    outer = np.concatenate([np.arange(i), np.arange(i + 1, len(Yt))])
    inner = loocv_curve(Xt[outer], Yt[outer], grid)
    lam_hat = min(inner, key=inner.get)
    w, b = fit_ridge(Xt[outer], Yt[outer], lam_hat)
    nested.append(score(w, b, Xt[i:i+1], Yt[i:i+1]))
    picks.append(lam_hat)
nested_est = float(np.mean(nested))

print(f"노이즈 플로어 (진짜 선형에 대한 잡음 분산) = {noise_var:.3f}")
print(f"(a) 단일 7/3 분할      : {best_err:.3f}  (best lambda={best_lam})")
print(f"(b) LOOCV 곡선의 min   : {loocv_best:.3f}   {curve}")
print(f"(c) nested LOOCV       : {nested_est:.3f}")
print(f"내부에서 lambda를 고른 결과: {picks}")

노이즈 플로어 (진짜 선형에 대한 잡음 분산) = 0.956
(a) 단일 7/3 분할      : 1.477  (best lambda=0.0)
(b) LOOCV 곡선의 min   : 1.514   {0.0: 1.5297026885081166, 1.0: 1.5142551285165713, 10.0: 1.9896729520178376}
(c) nested LOOCV       : 1.588
내부에서 lambda를 고른 결과: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0]


In [10]:
fig, ax = plt.subplots(figsize=(7, 4))
labels = ["노이즈 플로어\n(상한선 아님,\n노이즈 기준)", "단일 7/3 분할\n(선택 포함)", "LOOCV 곡선 min\n(선택 포함, 낙관적)", "nested LOOCV\n(선택 격리)"]
vals = [noise_var, best_err, loocv_best, nested_est]
bars = ax.bar(labels, vals, color=["0.6", "#4878a8", "#d95f02", "#1a9641"], width=0.6)
for b_, v in zip(bars, vals):
    ax.text(b_.get_x() + b_.get_width() / 2, v + 0.05, f"{v:.3f}", ha="center", fontsize=10)
ax.set_ylabel("MSE")
ax.set_title("'최종 성능' 추정치: 최적화 단계가 평가에 닿으면 낙관적 편향")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(IMG + "/ch06_2_nested_cv.svg")
plt.show()
print("figure saved -> kor/src/images/ch06_2_nested_cv.svg")

figure saved -> kor/src/images/ch06_2_nested_cv.svg


## 정리: 다음으로

- **6.3절 (train/val/test 분리)**: 이 노트북의 (b)/(c) 비교가 극한으로
  가면 — 검증셋으로 \(\lambda\)를 고르고 **그 검증셋으로** 최종 성능을
  주장하는 것 — 6.3절에서 다룰 "테스트 데이터를 몰래 본" 부정행위와
  같은 구조입니다. test를 완전히 격리해 둔 마지막 분할이 왜 필요한지
  6.3절에서 다룹니다.
- 실전에서는 이 노트북의 `cv_errors` 대신 `sklearn.model_selection.
  KFold(shuffle=True)` / `StratifiedKFold`를 쓰되, **분할의 순서를
  고정(seed)**하면 이 노트북의 숫자를 그대로 재현할 수 있습니다.